# 05 — Análise das perguntas de negócio

As consultas sobre as tabelas Gold respondem às perguntas por UF do cliente, mês da compra e quartil do frete total do pedido. Cada taxa usa o mesmo indicador persistido na fato: pedidos delivered com datas real e prevista disponíveis; atraso ocorre quando a data civil real é posterior à prevista.

As tabelas exibem numerador e denominador por grupo. O frete é uma comparação descritiva: distância, modalidade, peso e rota não são controlados.


In [ ]:
from pyspark.sql import functions as F
import re

dbutils.widgets.text("catalogo", "")
catalogo = dbutils.widgets.get("catalogo").strip() or spark.sql("SELECT current_catalog()").first()[0]
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalogo), "Informe um catálogo válido no widget."
assert catalogo != "hive_metastore", "Use o catálogo do workspace com Unity Catalog."
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
origem = f"/Volumes/{catalogo}/olist_mvp_raw/originais"
silver = f"{catalogo}.olist_mvp_silver"
gold = f"{catalogo}.olist_mvp_gold"
arquivos = {
    "pedidos": "olist_orders_dataset.csv",
    "itens": "olist_order_items_dataset.csv",
    "clientes": "olist_customers_dataset.csv",
}
colunas_esperadas = {
    "pedidos": ["order_id", "customer_id", "order_status", "order_purchase_timestamp",
                "order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date", "order_estimated_delivery_date"],
    "itens": ["order_id", "order_item_id", "product_id", "seller_id",
              "shipping_limit_date", "price", "freight_value"],
    "clientes": ["customer_id", "customer_unique_id", "customer_zip_code_prefix",
                 "customer_city", "customer_state"],
}
print(f"Catálogo: {catalogo}; volume: {origem}")


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

raizes = [Path.cwd().parent / "sql", Path.cwd() / "sql"]
diretorio_sql = next((raiz for raiz in raizes if (raiz / "atraso_por_uf.sql").exists()), None)
assert diretorio_sql is not None, "Abra o notebook pela Git folder com o diretório sql."
tabelas = {
    "fato": f"{gold}.fato_pedidos",
    "localidade": f"{gold}.dim_localidade",
    "datas": f"{gold}.dim_data",
}
def executar(nome):
    """Executa uma consulta do diretório sql com tabelas Gold qualificadas."""
    texto = (diretorio_sql / nome).read_text(encoding="utf-8")
    return spark.sql(texto.format(**tabelas))

fato = spark.table(tabelas["fato"])
# O total global reconcilia os denominadores das três perguntas.
resumo = fato.agg(
    F.count("*").alias("pedidos_total"),
    F.sum("pedido_entregue_com_datas").alias("pedidos_elegiveis"),
    F.sum("entrega_atrasada").alias("pedidos_atrasados"),
    F.min("data_compra").alias("primeira_compra"),
    F.max("data_compra").alias("ultima_compra"),
)
display(resumo)


## 1. Como varia a taxa de atraso por estado do cliente?

A UF vem da dimensão de localidade do cliente. A consulta mantém um grupo Sem UF para referências ausentes e mostra pedidos totais, elegíveis, atrasados e taxa, preservando o tamanho da amostra.


In [ ]:
por_uf = executar("atraso_por_uf.sql")
display(por_uf)
assert por_uf.agg(F.sum("pedidos_elegiveis")).first()[0] == resumo.first()["pedidos_elegiveis"]
# O gráfico usa as maiores amostras; a tabela mantém todas as UFs.
# A taxa é convertida para double apenas na visualização, pois Matplotlib usa float.
dados_uf = (por_uf.orderBy(F.desc("pedidos_elegiveis")).limit(10)
            .withColumn("taxa_atraso_pct", F.col("taxa_atraso_pct").cast("double"))
            .toPandas())
dados_uf = dados_uf.sort_values("taxa_atraso_pct")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(dados_uf["estado"], dados_uf["taxa_atraso_pct"], color="#236a8a")
for posicao, linha in enumerate(dados_uf.itertuples()):
    ax.text(linha.taxa_atraso_pct + 0.2, posicao, f"n={linha.pedidos_elegiveis}", va="center")
ax.set_title("Taxa de atraso por UF do cliente — 10 maiores amostras")
ax.set_xlabel("Pedidos atrasados / pedidos elegíveis (%)")
ax.set_ylabel("UF")
ax.set_xlim(0, max(100, dados_uf["taxa_atraso_pct"].max() + 10))
plt.tight_layout()
plt.show()
plt.close(fig)


AL tem a maior taxa observada, 85/397 = 21,41%, mas sua amostra é pequena. Entre UFs com pelo menos 1.000 elegíveis, CE registra 176/1.279 = 13,76%. RJ combina taxa de 12,11% com 12.350 elegíveis; SP tem 4,49% entre 40.494 elegíveis. O gráfico destaca as dez UFs com maiores amostras, enquanto a tabela conserva todas as 27. A UF representa o cliente, não a origem do envio.


## 2. Como evolui a taxa de atraso por mês da compra?

O agrupamento usa o mês civil da compra. A cobertura mede a fração dos pedidos do mês que já pertence à população elegível, necessária para interpretar meses recentes ou incompletos.


In [ ]:
por_mes = executar("atraso_por_mes.sql")
display(por_mes)
assert por_mes.agg(F.sum("pedidos_elegiveis")).first()[0] == resumo.first()["pedidos_elegiveis"]
# Taxas nulas continuam sem ponto no gráfico; não são convertidas em zero.
dados_mes = (por_mes.filter(F.col("ano_mes") != "Sem data").orderBy("ano_mes")
             .withColumn("taxa_atraso_pct", F.col("taxa_atraso_pct").cast("double"))
             .toPandas())
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dados_mes["ano_mes"], dados_mes["taxa_atraso_pct"], marker="o", color="#236a8a")
ax.set_title("Atraso por mês da compra — pedidos entregues com duas datas")
ax.set_ylabel("Pedidos atrasados / pedidos elegíveis (%)")
ax.set_xlabel("Mês da compra")
ax.tick_params(axis="x", rotation=70)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()
plt.close(fig)


In [ ]:
primeira = resumo.first()["primeira_compra"]
ultima = resumo.first()["ultima_compra"]
print(f"Intervalo observado de compras: {primeira} a {ultima}.")
if primeira.day != 1:
    print("O primeiro mês não começa no dia 1: período inicial incompleto.")
import calendar
if ultima.day != calendar.monthrange(ultima.year, ultima.month)[1]:
    print("O último mês não termina no último dia: período final incompleto.")


Março de 2018 registra 1.328 atrasos entre 7.003 elegíveis (18,96%); junho registra 71 entre 6.096 (1,16%). A cobertura de elegibilidade é 97,12% e 98,85%, respectivamente. Setembro de 2016 tem apenas um elegível e aparece com 100% de atraso, o que amplia a escala do gráfico sem representar uma tendência. Setembro e outubro de 2018 não têm pedidos elegíveis; suas taxas permanecem indefinidas. A série é descritiva e não identifica causas das oscilações.


## 3. Como a taxa se distribui por faixa de frete total do pedido?

O frete já foi somado por pedido na Gold. A consulta distribui os pedidos elegíveis com frete não negativo em quatro grupos de tamanho semelhante e apresenta fretes ausentes ou negativos separadamente.


In [ ]:
# Os quartis usam frete somado por pedido e o mesmo denominador global.
por_frete = executar("atraso_por_frete.sql")
display(por_frete)
assert por_frete.agg(F.sum("pedidos_elegiveis")).first()[0] == resumo.first()["pedidos_elegiveis"]
dados_frete = (por_frete.withColumn("taxa_atraso_pct", F.col("taxa_atraso_pct").cast("double"))
               .toPandas())
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(dados_frete["faixa_frete"], dados_frete["taxa_atraso_pct"], color="#8a5a23")
for posicao, linha in enumerate(dados_frete.itertuples()):
    ax.text(posicao, linha.taxa_atraso_pct + 0.2, f"n={linha.pedidos_elegiveis}",
            ha="center", fontsize=9)
ax.set_title("Atraso por quartil de frete do pedido — amostra elegível")
ax.set_ylabel("Pedidos atrasados / pedidos elegíveis (%)")
ax.set_xlabel("Faixa de frete total")
ax.tick_params(axis="x", rotation=25)
ax.set_ylim(0, max(100, dados_frete["taxa_atraso_pct"].max() + 10))
plt.tight_layout()
plt.show()
plt.close(fig)


Os quartis ordenam pedidos elegíveis com frete total não negativo por valor e order_id. Cada grupo contém cerca de 24 mil pedidos. A taxa vai de 4,40% no Q1 a 7,96% no Q4. Empates podem colocar o mesmo valor de frete em quartis vizinhos; por isso os limites monetários da tabela se sobrepõem. A diferença observada é uma associação, sem evidência de que o valor do frete cause atraso.


## Síntese da análise

A reconciliação final compara o total da fato, a população elegível e os atrasados. O indicador global é uma referência comum para as três segmentações, não a média simples das taxas de grupos.


In [ ]:
total = resumo.first()
taxa_geral = (100.0 * total["pedidos_atrasados"] / total["pedidos_elegiveis"]
              if total["pedidos_elegiveis"] else None)
print(f"Pedidos totais: {total['pedidos_total']}; elegíveis: {total['pedidos_elegiveis']}; "
      f"atrasados: {total['pedidos_atrasados']}; taxa geral: {taxa_geral:.2f}%."
      if taxa_geral is not None else "Nenhum pedido elegível; a taxa não pode ser calculada.")
